In [3]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta
import scipy.stats as stats

In [4]:
fake = Faker()
Faker.seed(33)
random.seed(33)

In [5]:
def get_customer_location(order_date):
    import random
    year = pd.to_datetime(order_date).year
    
    # as the business digitally matures
    if year <= 2021:
        ma_ratio = 0.40
    elif year == 2022:
        ma_ratio = 0.35
    elif year == 2023:
        ma_ratio = 0.32
    else:
        ma_ratio = 0.28

    raw_weights = {
        "CA": 0.14,
        "NY": 0.12,
        "FL": 0.09,
        "TX": 0.08,
        "IL": 0.06,
        "NJ": 0.05,
        "WA": 0.04,
        "VA": 0.04,
        "PA": 0.03,
        "GA": 0.03,
        "NC": 0.03,
        "AZ": 0.02,
        "CO": 0.02
    }

    total = sum(raw_weights.values())
    states_distribution = [("MA", ma_ratio)] + [
        (state, (1 - ma_ratio) * weight / total)
        for state, weight in raw_weights.items()
    ]

    states, weights = zip(*states_distribution)
    return random.choices(states, weights=weights, k=1)[0]

# Generate Customer Data with Dynamic Location Assignment
def generate_customers(num_customers=1150):
    customers = []
    for i in range(1, num_customers + 1):
        signup_date = fake.date_between(start_date=datetime(2019, 1, 1).date(), end_date=datetime(2024, 12, 31).date())
        state = get_customer_location(signup_date)

        if state == "MA":
            location = state
        else:
            location = state

        customers.append([
            i,
            fake.name(),
            fake.email(),
            signup_date,
            location
        ])

    return pd.DataFrame(customers, columns=["customer_id", "name", "email", "signup_date", "location"])


In [6]:
# Generate Customers Data
customers_df = generate_customers()

In [7]:
# product %; low boundary; high boundary; revenue %
subcategory_distribution = {
    "Cardigan": (28, 200, 600, 18),
    "Handbags": (5, 800, 1200, 17.6),
    "Bracelets": (2, 500, 800, 9),
    "Pants": (8, 300, 500, 8.2),
    "Shirts & Tops": (25, 250, 450, 17.6),
    "Coats & Jackets": (18, 700, 1000, 15.3),
    "Scarf": (8, 200, 600, 9.3),
    "Skirts": (5, 300, 500, 5),
    "Charms & Pendants": (2, 500, 800, 5)
}
# Original stock
original_stocking = {
    "Cardigan": 20,
    "Handbags": 5,
    "Bracelets": 5,
    "Pants": 10,
    "Shirts & Tops": 30,
    "Coats & Jackets": 15,
    "Scarf": 5,
    "Skirts": 10,
    "Charms & Pendants": 5
}

# Define customer purchase frequency based on research
purchase_frequency = {
    "Cardigan": 2.5,
    "Handbags": 1.2,
    "Bracelets": 2.8,
    "Pants": 3.0,
    "Shirts & Tops": 4.5,
    "Coats & Jackets": 1.8,
    "Scarf": 2.5,
    "Skirts": 2.0,
    "Charms & Pendants": 2.2
}

# Define stocking months based on subcategory
stocking_months = {
    "Cardigan": [9, 10, 11, 12, 1, 2, 3],
    "Pants": [1, 2, 3, 4, 5, 9, 10, 11, 12],
    "Shirts & Tops": list(range(1, 13)),
    "Coats & Jackets": [9, 10, 11, 12, 1, 2, 3],
    "Skirts": [3, 4, 5, 6, 7, 8, 9, 12],
    "Handbags": list(range(1, 13)),
    "Bracelets": list(range(1, 13)),
    "Scarf": list(range(1, 13)),
    "Charms & Pendants": list(range(1, 13))
}

In [8]:
def generate_price(min_val, max_val):
    std_dev = (max_val - min_val) / 6
    mean = (max_val + min_val) / 2
    price = np.clip(np.random.normal(mean, std_dev), min_val, max_val)
    return round(price, 0)

def generate_cost(price):
    return round(price * 0.6, 0)

In [9]:
# Product by sub-category
total_products = 350
subcategory_counts = {k: int(v[0] / 100 * total_products) for k, v in subcategory_distribution.items()}
subcategory_counts

{'Cardigan': 98,
 'Handbags': 17,
 'Bracelets': 7,
 'Pants': 28,
 'Shirts & Tops': 87,
 'Coats & Jackets': 63,
 'Scarf': 28,
 'Skirts': 17,
 'Charms & Pendants': 7}

In [10]:
# Generate Products Data with Normal Distribution for Price
products = []
for subcategory, (percentage, min_price, max_price, revenue_share) in subcategory_distribution.items():
    count = subcategory_counts[subcategory]
#     print(count)
    for _ in range(count):
        category = "Clothing" if subcategory in ["Cardigan", "Pants", "Shirts & Tops", "Coats & Jackets", "Skirts"] else "Accessories"
        price = generate_price(min_price, max_price)
        cost = generate_cost(price)
        products.append([
            len(products) + 1, category, subcategory, price, cost
        ])
products_df = pd.DataFrame(products, columns=["product_id", "category", "subcategory", "price", "cost"])
products_df

,product_id,category,subcategory,price,cost
0,1,Clothing,Cardigan,373.0,224.0
1,2,Clothing,Cardigan,424.0,254.0
2,3,Clothing,Cardigan,498.0,299.0
3,4,Clothing,Cardigan,395.0,237.0
4,5,Clothing,Cardigan,520.0,312.0
...,...,...,...,...,...
347,348,Accessories,Charms & Pendants,670.0,402.0
348,349,Accessories,Charms & Pendants,664.0,398.0
349,350,Accessories,Charms & Pendants,628.0,377.0
350,351,Accessories,Charms & Pendants,547.0,328.0


In [11]:
inventory = []
for _, row in products_df.iterrows():
    subcategory = row["subcategory"]
    avg_stock = max(1, int(original_stocking[subcategory]))  # Ensure no zero stock
    
    # Adjust stock based on price relative to subcategory mean
    subcategory_mean_price = ((subcategory_distribution[subcategory][2] + subcategory_distribution[subcategory][3])) / 2
    if row["price"] < subcategory_mean_price:
        avg_stock = int(avg_stock * 1.2)  # Slightly more inventory for lower-priced items
    else:
        avg_stock = int(avg_stock * 0.8)  # Slightly less inventory for higher-priced items
    
    avg_stock = max(1, avg_stock)  # Ensure no zero inventory
    
    # Assign original stocking date (reduce probability for 2020-2021 due to pandemic)
    year = random.choices([2019, 2020, 2021, 2022, 2023, 2024], weights=[1, 0.5, 0.5, 1, 1, 1])[0]
    month = random.choice(stocking_months[subcategory])
    original_stocking_date = datetime(year, month, random.randint(1, 28)).date()
    
    # Define sizes based on subcategory
    if subcategory in ["Bracelets", "Scarf", "Charms & Pendants","Handbags"]:
        inventory.append([row["product_id"], original_stocking_date, avg_stock])
    else:
        sizes = ["XS", "S", "M", "L", "XL"]
        stock_levels_by_size = [max(1, avg_stock // 8), max(1, avg_stock // 4), max(1, avg_stock // 4), max(1, avg_stock // 4), max(1, avg_stock // 8)]
        stock_levels         = [np.sum(stock_levels_by_size)] + stock_levels_by_size
        inventory.append([row["product_id"], original_stocking_date, *stock_levels])

# Convert inventory to DataFrame and merge
columns = ["product_id", "Stocking_Date","Total_Inventory", "XS_stock", "S_stock", "M_stock", "L_stock", "XL_stock"]
inventory_df = pd.DataFrame(inventory, columns=columns[:len(inventory[0])])
products_df = products_df.merge(inventory_df, on="product_id")
products_df.insert(5, "Inventory_Batch", 1)

In [12]:
products_df

,product_id,category,subcategory,price,cost,Inventory_Batch,Stocking_Date,Total_Inventory,XS_stock,S_stock,M_stock,L_stock,XL_stock
0,1,Clothing,Cardigan,373.0,224.0,1,2022-10-02,16,2.0,4.0,4.0,4.0,2.0
1,2,Clothing,Cardigan,424.0,254.0,1,2019-12-16,16,2.0,4.0,4.0,4.0,2.0
2,3,Clothing,Cardigan,498.0,299.0,1,2023-09-23,16,2.0,4.0,4.0,4.0,2.0
3,4,Clothing,Cardigan,395.0,237.0,1,2022-12-12,16,2.0,4.0,4.0,4.0,2.0
4,5,Clothing,Cardigan,520.0,312.0,1,2024-12-06,16,2.0,4.0,4.0,4.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,348,Accessories,Charms & Pendants,670.0,402.0,1,2019-10-15,4,NaN,NaN,NaN,NaN,NaN
348,349,Accessories,Charms & Pendants,664.0,398.0,1,2021-12-21,4,NaN,NaN,NaN,NaN,NaN
349,350,Accessories,Charms & Pendants,628.0,377.0,1,2019-06-28,4,NaN,NaN,NaN,NaN,NaN
350,351,Accessories,Charms & Pendants,547.0,328.0,1,2020-03-22,4,NaN,NaN,NaN,NaN,NaN


Features wanted: order_id, customer_id, order_date, product_id, size, status, fulfillment. And order data should satisfy requests below:
1. Total revenue and subcategory revenue percentages match the given targets.
1a. Total revenue is around 1,500,000 per year
1b. cardigan takes up 18% of total revenue, Handbags takes 17.6%, Bracelets take 9%, Pants take 8.2%, Shirts & Tops take 17.6%, Coats & Jackets take 15.3%, scarf takes 9.3%, skirts take 5%
2. Customers cannot place orders before their sign-up date.
3. Products are only available if the original stocking date is within one year of the order date.
4. There are 3 status which are 'Prepaid', 'Completed', and 'Returned'
4a. A 5% return rate is applied, with slightly higher returns after mid-year and holiday seasons.
4b. Returned orders doesn't reduce inventory. And fulfillment is True
4c. If a product is out of stock, it is marked as 'Prepaid'. After 3 out-of-stock requests for a product, a new row would be created in product table with Clothing restocks 5 per size, while Accessories restock 5 units total, and the inventory_batch should be added by 1. Then after restocking, go back to order data, check where product_id and size match, then mark fulfillment as True
4d. If a product is not out of stock, it is marked as 'Completed'. And fulfillment is True
5. Our Cardigan, Shirts & Tops, Coats & Jackets are popular. And most of the people would be fit in s,m,l

### Basic

In [197]:
# # Assume products_df is defined globally before calling generate_orders
# global products_df

# # Generate Order Dates with a Seasonal Trend
# def generate_order_dates(num_orders=20000):
#     date_range = pd.date_range(start="2019-01-01", end="2024-12-31", freq="D")
#     weights = np.sin(np.linspace(0, 6 * np.pi, len(date_range))) + 2  # Seasonal trend
    
#     # Adjust weights for the pandemic period (2020-2021) to reduce order frequency
#     pandemic_start = pd.Timestamp("2020-03-01")
#     pandemic_end = pd.Timestamp("2021-9-30")
#     for i, date in enumerate(date_range):
#         if pandemic_start <= date <= pandemic_end:
#             weights[i] *= 0.5  # Reduce orders by 50% during pandemic
    
#     weights = weights / weights.sum()  # Normalize weights
#     return np.sort(np.random.choice(date_range, num_orders, p=weights))

# # Select valid product
# def select_valid_product(subcategory, order_date):
#     latest_batch = products_df.groupby("product_id")["Inventory_Batch"].max().reset_index()
#     products_df_latest = products_df.merge(latest_batch, on=["product_id", "Inventory_Batch"], how="inner")
#     available_products = products_df_latest[(products_df_latest["subcategory"] == subcategory) & (pd.to_datetime(products_df_latest["Stocking_Date"]) <= order_date)] 
#                                             #& (pd.to_datetime(products_df_latest["Stocking_Date"]) + pd.DateOffset(years=1) >= order_date)]
#     if not available_products.empty:
#         return available_products.sample(1).iloc[0]
#     return None

# # Restock function
# def restock_product(product, order_date):
#     latest_inventory = products_df.loc[products_df["product_id"] == product["product_id"]].nlargest(1, "Inventory_Batch")["Total_Inventory"].values[0]
    
#     if product["category"] == "Clothing":
#         size_columns = ["XS_stock", "S_stock", "M_stock", "L_stock", "XL_stock"]
#         current_stock = products_df.loc[products_df["product_id"] == product["product_id"]].nlargest(1, "Inventory_Batch")[size_columns].values[0]
#         restock_amount = [max(0, 3 - stock) if stock < 3 else 0 for stock in current_stock]  # Top up to 3/ left cloths
#     else:
#         restock_amount = max(0, 3 - latest_inventory)  # Top up to 3 for accessories
    
#     new_batch = product["Inventory_Batch"] + 1
#     new_product = product.copy()
#     new_product["Inventory_Batch"] = new_batch
#     new_product["Stocking_Date"] = order_date
    
#     if product["category"] == "Clothing":
#         for idx, col in enumerate(size_columns):
#             new_product[col] = max(0, current_stock[idx] + restock_amount[idx])
#         new_product["Total_Inventory"] = sum(new_product[size_columns])
#     else:
#         new_product["Total_Inventory"] = latest_inventory + restock_amount
    
#     return new_product

# # Generate Orders Data
# def generate_orders(num_orders=30000):
#     global products_df  # Ensure we modify the global products_df
    
#     order_dates = generate_order_dates(num_orders)
#     orders = []
#     out_of_stock_tracker = {}
#     restock_log = {}  # {(product_id, order_date): batch_number}
    
#     for i, order_date in enumerate(order_dates, 1):
#         customer = customers_df.sample(1).iloc[0]
#         if order_date < pd.to_datetime(customer["signup_date"]):
#             continue
        
#         subcategory = random.choices(list(subcategory_distribution.keys()), weights=[v[3] for v in subcategory_distribution.values()])[0]
#         product = select_valid_product(subcategory, order_date)
#         if product is None:
#             continue
        
#         product_id, category = product["product_id"], product["category"]
#         sizes = ["XS", "S", "M", "L", "XL"]
#         size = random.choices(sizes, weights=[0.1, 0.3, 0.3, 0.2, 0.1])[0] if category == "Clothing" else None
        
#          # Check inventory before updating
#         latest_batch_idx = products_df[products_df["product_id"] == product_id]["Inventory_Batch"].idxmax()
#         first_batch_idx = products_df[products_df["product_id"] == product_id]["Inventory_Batch"].idxmin()
                
#         if category == "Accessories" or products_df.at[latest_batch_idx, f"{size}_stock"] > 0:
#             status, fulfillment = "Completed", True
#             products_df.at[latest_batch_idx, "Total_Inventory"] -= 1
#             if category == "Clothing":
#                 products_df.at[latest_batch_idx, f"{size}_stock"] -= 1
        
#         else:
#             out_of_stock_tracker[product_id] = out_of_stock_tracker.get(product_id, 0) + 1
#             status, fulfillment = "Requested", False
#             if out_of_stock_tracker[product_id] >= 3 and (product_id, order_date) not in restock_log and (order_date >= pd.to_datetime(products_df.at[latest_batch_idx, 'Stocking_Date']) + pd.DateOffset(months=10)):
#                 new_product = restock_product(product, order_date)
#                 products_df = pd.concat([products_df, pd.DataFrame([new_product])], ignore_index=True)
#                 restock_log[(product_id, order_date)] = new_product["Inventory_Batch"]
#                 out_of_stock_tracker[product_id] = 0
#                 fulfillment = True
        
# #         # Ensure Total_Inventory consistency
# #         if category == "Clothing":
# #             products_df.at[latest_batch_idx, "Total_Inventory"] = products_df.loc[latest_batch_idx, ["XS_stock", "S_stock", "M_stock", "L_stock", "XL_stock"]].sum()
        
#         # Handle returns
#         return_rate = 0.07 if pd.Timestamp(order_date).month in [7, 8, 1] else 0.05
#         if (random.random() < return_rate) & (fulfillment == True):
#             status, fulfillment = "Returned", True
#             products_df.at[latest_batch_idx, "Total_Inventory"] += 1
#             if category == "Clothing":
#                 products_df.at[latest_batch_idx, f"{size}_stock"] += 1
        
#         orders.append([i, customer["customer_id"], order_date, product_id, size, status, fulfillment])
    
#     return pd.DataFrame(orders, columns=["order_id", "customer_id", "order_date", "product_id", "size", "status", "fulfillment"])

# # Generate orders and store in DataFrame
# orders_df = generate_orders()

# # Show a preview of the generated data
# print("Orders Sample:")
# print(orders_df.head())

    
    

### mutiple items within one order

In [198]:
# # Assume products_df is defined globally before calling generate_orders
# global products_df

# # Generate Order Dates with a Seasonal Trend
# def generate_order_dates(num_orders=16000):
#     date_range = pd.date_range(start="2019-01-01", end="2024-12-31", freq="D")
#     weights = np.sin(np.linspace(0, 6 * np.pi, len(date_range))) + 2  # Seasonal trend
    
#     # Adjust weights for the pandemic period (2020-2021) to reduce order frequency
#     pandemic_start = pd.Timestamp("2020-03-01")
#     pandemic_end = pd.Timestamp("2021-9-30")
#     for i, date in enumerate(date_range):
#         if pandemic_start <= date <= pandemic_end:
#             weights[i] *= 0.5  # Reduce orders by 50% during pandemic
    
#     weights = weights / weights.sum()  # Normalize weights
#     return np.sort(np.random.choice(date_range, num_orders, p=weights))

# # Select valid product
# def select_valid_product(subcategory, order_date):
#     latest_batch = products_df.groupby("product_id")["Inventory_Batch"].max().reset_index()
#     products_df_latest = products_df.merge(latest_batch, on=["product_id", "Inventory_Batch"], how="inner")
#     available_products = products_df_latest[(products_df_latest["subcategory"] == subcategory) & (pd.to_datetime(products_df_latest["Stocking_Date"]) <= order_date)] 
#                                             #& (pd.to_datetime(products_df_latest["Stocking_Date"]) + pd.DateOffset(years=1) >= order_date)]
#     if not available_products.empty:
#         return available_products.sample(1).iloc[0]
#     return None

# # Restock function
# def restock_product(product, order_date):
#     latest_inventory = products_df.loc[products_df["product_id"] == product["product_id"]].nlargest(1, "Inventory_Batch")["Total_Inventory"].values[0]
    
#     if product["category"] == "Clothing":
#         size_columns = ["XS_stock", "S_stock", "M_stock", "L_stock", "XL_stock"]
#         current_stock = products_df.loc[products_df["product_id"] == product["product_id"]].nlargest(1, "Inventory_Batch")[size_columns].values[0]
#         restock_amount = [max(0, 3 - stock) if stock < 3 else 0 for stock in current_stock]  # Top up to 3/ left cloths
#     else:
#         restock_amount = max(0, 3 - latest_inventory)  # Top up to 3 for accessories
    
#     new_batch = product["Inventory_Batch"] + 1
#     new_product = product.copy()
#     new_product["Inventory_Batch"] = new_batch
#     new_product["Stocking_Date"] = order_date
    
#     if product["category"] == "Clothing":
#         for idx, col in enumerate(size_columns):
#             new_product[col] = max(0, current_stock[idx] + restock_amount[idx])
#         new_product["Total_Inventory"] = sum(new_product[size_columns])
#     else:
#         new_product["Total_Inventory"] = latest_inventory + restock_amount
    
#     return new_product

# # Generate Orders Data
# def generate_orders(num_orders=16000):
#     global products_df  # Ensure we modify the global products_df
#     order_dates = generate_order_dates(num_orders)
#     orders = []
#     out_of_stock_tracker = {}
#     restock_log = {}  # {(product_id, order_date): batch_number}
    
#     for i, order_date in enumerate(order_dates, 1):
#         num_items = random.choices([1, 2, 3, 4, 5], weights=[0.5, 0.3, 0.1, 0.08, 0.02])[0]  # Randomly select number of items per order
#         customer = customers_df.sample(1).iloc[0]
#         if order_date < pd.to_datetime(customer["signup_date"]):
#             continue
        
#         subcategory = random.choices(list(subcategory_distribution.keys()), weights=[v[3] for v in subcategory_distribution.values()])[0]
#         for _ in range(num_items):
#             subcategory = random.choices(list(subcategory_distribution.keys()), weights=[v[3] for v in subcategory_distribution.values()])[0]
#             product = select_valid_product(subcategory, order_date)
#             if product is None:
#                 continue
            
#             product_id, category = product["product_id"], product["category"]
#             sizes = ["XS", "S", "M", "L", "XL"]
#             size = random.choices(sizes, weights=[0.1, 0.3, 0.3, 0.2, 0.1])[0] if category == "Clothing" else None

#              # Check inventory before updating
#             latest_batch_idx = products_df[products_df["product_id"] == product_id]["Inventory_Batch"].idxmax()
#             first_batch_idx = products_df[products_df["product_id"] == product_id]["Inventory_Batch"].idxmin()

#             if category == "Accessories" and products_df.at[latest_batch_idx, "Total_Inventory"] > 0:
#                 status, fulfillment = "Completed", True
#                 products_df.at[latest_batch_idx, "Total_Inventory"] -= 1
#             elif category == "Clothing" and products_df.at[latest_batch_idx, f"{size}_stock"] > 0:
#                 status, fulfillment = "Completed", True
#                 products_df.at[latest_batch_idx, "Total_Inventory"] -= 1
#                 products_df.at[latest_batch_idx, f"{size}_stock"] -= 1

#             else:
#                 out_of_stock_tracker[product_id] = out_of_stock_tracker.get(product_id, 0) + 1
#                 status, fulfillment = "Requested", False
#                 if out_of_stock_tracker[product_id] >= 3 and (product_id, order_date) not in restock_log and (order_date <= pd.to_datetime(products_df.at[first_batch_idx, 'Stocking_Date']) + pd.DateOffset(months=10)):
#                     new_product = restock_product(product, order_date)
#                     products_df = pd.concat([products_df, pd.DataFrame([new_product])], ignore_index=True)
#                     restock_log[(product_id, order_date)] = new_product["Inventory_Batch"]
#                     out_of_stock_tracker[product_id] = 0
# #                     fulfillment = True

#     #         # Ensure Total_Inventory consistency
#     #         if category == "Clothing":
#     #             products_df.at[latest_batch_idx, "Total_Inventory"] = products_df.loc[latest_batch_idx, ["XS_stock", "S_stock", "M_stock", "L_stock", "XL_stock"]].sum()

#             # Handle returns
#             return_rate = 0.07 if pd.Timestamp(order_date).month in [7, 8, 1] else 0.05
#             if (random.random() < return_rate) & (fulfillment == True):
#                 status, fulfillment = "Returned", True
#                 products_df.at[latest_batch_idx, "Total_Inventory"] += 1
#                 if category == "Clothing":
#                     products_df.at[latest_batch_idx, f"{size}_stock"] += 1

#             orders.append([i, customer["customer_id"], order_date, product_id, size, status, fulfillment])
    
#     return pd.DataFrame(orders, columns=["order_id", "customer_id", "order_date", "product_id", "size", "status", "fulfillment"])

# # Generate orders and store in DataFrame
# orders_df = generate_orders(num_orders= 16000)

# # Show a preview of the generated data
# print("Orders Sample:")
# print(orders_df.head())


### fulfill requested order

In [13]:
# Assume products_df is defined globally before calling generate_orders
global products_df

# Track customers waiting for restocked products
waiting_customers = {}

# Generate Order Dates with a Seasonal Trend
def generate_order_dates(num_orders=16000):

    date_range = pd.date_range(start="2019-01-01", end="2024-12-31", freq="D")
    num_days = len(date_range)
    weights = np.ones(num_days) * 0.5  # Increased baseline weight to balance early years  # Baseline weight to prevent zero orders
    
    # Define Gaussian peaks for mid-year (June-July) and end-year (November-December)
    mid_year_peak = stats.norm.pdf(range(num_days), loc=num_days * 8/12, scale=num_days * 1.5/12)
    end_year_peak = stats.norm.pdf(range(num_days), loc=num_days * 11.5/12, scale=num_days * 1.5/12)
    
    # Combine the seasonal trends
    weights += (mid_year_peak * 2.5 + end_year_peak * 1.5)  # Increase mid-year peak strength even further in 2024  # Increase mid-year peak strength  # Boost early years to align with later trends
    
    # Adjust weights for the pandemic period (2020-2021) to reduce order frequency
    pandemic_start = pd.Timestamp("2020-03-01")
    pandemic_end = pd.Timestamp("2021-09-30")
    for i, date in enumerate(date_range):
        if pandemic_start <= date <= pandemic_end:
            weights[i] *= 0.7  # Reduce orders by 50% during pandemic
    
    # Ensure 2019 is similar to 2022
    for i, date in enumerate(date_range):
        if pd.Timestamp("2019-01-01") <= date < pd.Timestamp("2020-01-01"):
            weights[i] *= 1.8  # Ensure 2019 aligns with 2022 order volumes
            weights[i] *= 1.5  # Increase to better match 2022  # Adjust 2019 to be similar to 2022
        elif pd.Timestamp("2022-01-01") <= date < pd.Timestamp("2023-01-01"):
            weights[i] *= 1.02  # 20% increase from 2022
        elif pd.Timestamp("2023-01-01") <= date < pd.Timestamp("2024-01-01"):
            weights[i] *= 1.02  # 40% increase from 2023
        elif pd.Timestamp("2024-01-01") <= date < pd.Timestamp("2024-09-01"):
            weights[i] *= 1.02  # Increase 2024 H1 orders slightly to balance mid-year peak
    
    # Sharp increase for second half of 2024 due to customer segmentation
    segmentation_boost_start = pd.Timestamp("2024-09-01")
    segmentation_boost_end = pd.Timestamp("2024-12-31")
    for i, date in enumerate(date_range):
        if segmentation_boost_start <= date <= segmentation_boost_end:
            weights[i] *= 1.4  # Reduce the segmentation boost slightly to maintain mid-year peak
    
    weights /= weights.sum()  # Normalize after adjustments  # Normalize weights
    return np.sort(np.random.choice(date_range, num_orders, p=weights))

# Select valid product
def select_valid_product(subcategory, order_date):
    latest_batch = products_df.groupby("product_id")["Inventory_Batch"].max().reset_index()
    products_df_latest = products_df.merge(latest_batch, on=["product_id", "Inventory_Batch"], how="inner")
    available_products = products_df_latest[(products_df_latest["subcategory"] == subcategory) & (pd.to_datetime(products_df_latest["Stocking_Date"]) <= order_date)] 

    if not available_products.empty:
        return available_products.sample(1).iloc[0]
    return None

# Modify Restock function to trigger customer reorders
def restock_product(product, order_date, orders):

    if product["category"] == "Clothing":
        size_columns = ["XS_stock", "S_stock", "M_stock", "L_stock", "XL_stock"]
        current_stock = products_df.loc[products_df["product_id"] == product["product_id"]].nlargest(1, "Inventory_Batch")[size_columns].values[0]
        restock_amount = [max(0, 3 - stock) if stock < 3 else 0 for stock in current_stock]  # Top up to 3
    else:
        latest_inventory = products_df.loc[products_df["product_id"] == product["product_id"]].nlargest(1, "Inventory_Batch")["Total_Inventory"].values[0]
        restock_amount = max(0, 3 - latest_inventory)  # Top up to 3 for accessories
    
    new_batch = product["Inventory_Batch"] + 1
    new_product = product.copy()
    new_product["Inventory_Batch"] = new_batch
    new_product["Stocking_Date"] = order_date
    
    if product["category"] == "Clothing":
        for idx, col in enumerate(size_columns):
            new_product[col] = max(0, current_stock[idx] + restock_amount[idx])
        new_product["Total_Inventory"] = sum(new_product[size_columns])
    else:
        new_product["Total_Inventory"] = latest_inventory + restock_amount
    
    # Trigger customer reorders for 70% of those who requested
    if product["product_id"] in waiting_customers:
        for customer_id, size in waiting_customers[product["product_id"]]:
            if random.random() < 0.7:
                orders.append([len(orders) + 1, customer_id, order_date, product["product_id"], size, "Completed", True])
        del waiting_customers[product["product_id"]]  # Clear after restock attempt
    
    return new_product
# Modify order generation to track waiting customers
def generate_orders(num_orders=16000):
    global products_df  # Ensure we modify the global products_df
    order_dates = generate_order_dates(num_orders)
    orders = []
    out_of_stock_tracker = {}
    restock_log = {}  # {(product_id, order_date): batch_number}
    
    for i, order_date in enumerate(order_dates, 1):
        customer = customers_df.sample(1).iloc[0]
        num_items = random.choices([1, 2, 3, 4, 5], weights=[0.5, 0.3, 0.1, 0.08, 0.02])[0]  # Randomly select number of items per order
        if order_date < pd.to_datetime(customer["signup_date"]):
            continue
        
        for _ in range(num_items):
            subcategory = random.choices(list(subcategory_distribution.keys())
                                         , weights=[v[3] * (5 if v[0] > 5 else 1) for v in subcategory_distribution.values()]
                        )[0]
            product = select_valid_product(subcategory, order_date)
            if product is None:
                continue
            
            product_id, category = product["product_id"], product["category"]
            sizes = ["XS", "S", "M", "L", "XL"]
            size = random.choices(sizes, weights=[0.1, 0.3, 0.3, 0.2, 0.1])[0] if category == "Clothing" else None
            
            latest_batch_idx = products_df[products_df["product_id"] == product_id]["Inventory_Batch"].idxmax()
            first_batch_idx = products_df[products_df["product_id"] == product_id]["Inventory_Batch"].idxmin()
            
            if category == "Accessories" and products_df.at[latest_batch_idx, "Total_Inventory"] > 0:
                status, fulfillment = "Completed", True
                products_df.at[latest_batch_idx, "Total_Inventory"] -= 1
            elif category == "Clothing" and products_df.at[latest_batch_idx, f"{size}_stock"] > 0:
                status, fulfillment = "Completed", True
                products_df.at[latest_batch_idx, "Total_Inventory"] -= 1
                products_df.at[latest_batch_idx, f"{size}_stock"] -= 1
            else:
                out_of_stock_tracker[product_id] = out_of_stock_tracker.get(product_id, 0) + 1
                status, fulfillment = "Requested", False
                if product_id not in waiting_customers:
                    waiting_customers[product_id] = []
                waiting_customers[product_id].append((customer["customer_id"], size))
                
                if out_of_stock_tracker[product_id] >= 3 and (product_id, order_date) not in restock_log and (order_date <= pd.to_datetime(products_df.at[first_batch_idx, 'Stocking_Date']) + pd.DateOffset(months=10)):
                    new_product = restock_product(product, order_date,orders)
                    products_df = pd.concat([products_df, pd.DataFrame([new_product])], ignore_index=True)
                    restock_log[(product_id, order_date)] = new_product["Inventory_Batch"]
                    out_of_stock_tracker[product_id] = 0
            
#             Handle returns
            return_rate = 0.07 if pd.Timestamp(order_date).month in [7, 8, 1] else 0.05
            if (random.random() < return_rate) & (fulfillment == True):
                status, fulfillment = "Returned", True
                products_df.at[latest_batch_idx, "Total_Inventory"] += 1
                if category == "Clothing":
                    products_df.at[latest_batch_idx, f"{size}_stock"] += 1

            orders.append([i, customer["customer_id"], order_date, product_id, size, status, fulfillment])
    
    return pd.DataFrame(orders, columns=["order_id", "customer_id", "order_date", "product_id", "size", "status", "fulfillment"])

# Generate orders and store in DataFrame
orders_df = generate_orders(num_orders= 16000)

# Show a preview of the generated data
print("Orders Sample:")
print(orders_df.head())


Orders Sample:
   order_id  customer_id order_date  product_id size     status  fulfillment
0       162         1070 2019-01-12          73    L  Completed         True
1       475          783 2019-02-01         249    L  Completed         True
2       475          783 2019-02-01          67    S  Completed         True
3       532          839 2019-02-05         249   XL  Completed         True
4       605         1061 2019-02-09         249    M   Returned         True


In [14]:
orders_df.shape

(12744, 7)

In [15]:
orders_df.head()

,order_id,customer_id,order_date,product_id,size,status,fulfillment
0,162,1070,2019-01-12,73,L,Completed,True
1,475,783,2019-02-01,249,L,Completed,True
2,475,783,2019-02-01,67,S,Completed,True
3,532,839,2019-02-05,249,XL,Completed,True
4,605,1061,2019-02-09,249,M,Returned,True


In [16]:
q = orders_df.join(products_df[['product_id','price']], how = 'left', on='product_id', rsuffix='_2')
q['price'].sum()

6479730.0

In [17]:
q.shape

(12744, 9)

______

In [18]:
# Save to CSV
orders_df.to_csv("orders_data.csv", index=False)
customers_df.to_csv("customers_data.csv", index=False)
products_df.to_csv("products_data.csv", index=False)

In [20]:
customers_df.head(20)

,customer_id,name,email,signup_date,location
0,1,David Pittman,ipearson@example.net,2023-11-07,FL
1,2,Susan Harding,tammy87@example.com,2020-08-02,FL
2,3,Madison Turner,april09@example.net,2021-05-26,NJ
3,4,Ryan Reilly,davidbrown@example.com,2024-10-05,MA
4,5,Andrea Diaz DDS,williamflores@example.com,2020-10-24,FL
5,6,Kristen Price,suarezmichael@example.net,2023-06-19,PA
6,7,Becky Hernandez,mortonaaron@example.com,2024-10-22,GA
7,8,Sean Rhodes,eellis@example.com,2021-05-05,MA
8,9,Rachel Parker,smithnoah@example.com,2024-01-06,TX
9,10,Dana Giles,ssharp@example.org,2019-12-11,FL
